In [5]:
# =================================================================================================
# Raw data : This is Lab data from NHANES
# Purpose : Creating raw CSV file which converted from .xpt and generate ths Labs data.
# Specification : Each NHANE dataset has SEQN "Participent ID" which is uniqud id for labs. 
# Files Used :  L10AM_C.csv, L13_C.csv,PH_C.csv downloaded from Source 
# Data Dictinary : 
# Data Availability Matrix : 
# Source Site : 
# Developer Notes :  The notebook had created for windows desktop environment with Python3.
# File Creator : Neha Khairnar
# ===================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path
# # =================================================================================================
# 📂 Step 2: Paths below resolved for locally run datasets
BASE_DIR = Path().resolve().parent
#DATA_DIR = BASE_DIR / "data" / "dataset"/ "NHANE_Continuous_DemographicsData_2003_2004_Labs"
# # =================================================================================================
# 📥 Step 3: Load ALL Files (NO COLUMN DROPPING)
# # =================================================================================================

glucose_cpep_insulin_file ="../data/2003_2004/Labs/L10AM_C.csv"
cholesterol_file = "../data/2003_2004/Labs/L13_C.csv"
fasting_q_file = "../data/2003_2004/Labs/PH_C.csv"
LBXGH_file = "../data/2003_2004/Labs/L10_C.csv"

#BMI_file ="../data/2003_2004/BMX_C.csv"


glucose_cpep_insulin = pd.read_csv(glucose_cpep_insulin_file)
chol = pd.read_csv(cholesterol_file)
fasting = pd.read_csv(fasting_q_file)
LBXGH = pd.read_csv(LBXGH_file)

files = {
    "glucose_cpep_insulin_L10AM_C": glucose_cpep_insulin_file,
    "cholesterol_L13_C": cholesterol_file,
    "fasting_PH_C": fasting_q_file,
    "LBXGH_C" : LBXGH_file
}

# =================================================================================================
# 📊 Summary container uses above JSON for inspecting file purpose which
# # ===============================================================================================
summary = []
# =========================
# 🔁 Loop through files
# =========================
for name, path in files.items():
    df = pd.read_csv(path)
    summary.append({
        "file": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": list(df.columns)
    })

# =========================
# 📋 Convert to DataFrame
# =========================
summary_df = pd.DataFrame(summary)
# =========================
# 👀 Display summary
# =========================
print("📊 NHANES File Structure Summary:\n")
display(summary_df[["file", "rows", "columns", "column_names"]])

# =========================
# 🧠 Print full column lists
# =========================
for i, row in summary_df.iterrows():
    print("\n" + "="*50)
    print("FILE:", row["file"])
    print("COLUMNS:", len(row["column_names"]))
    print(row["column_names"])

# =========================
# 🧹 Step 4: Standardize Column Names
# =========================
glucose_cpep_insulin.columns = glucose_cpep_insulin.columns.str.upper()
chol.columns = chol.columns.str.upper()
fasting.columns = fasting.columns.str.upper()
LBXGH.columns = LBXGH.columns.str.upper()

# ===================================================================================================
# 🔗 Step 5: Merge ALL columns (FULL OUTER JOIN) which will have unique sequence numbers with all values
# ===================================================================================================
df = glucose_cpep_insulin.merge(
        chol,
        on="SEQN",
        how="outer",
        suffixes=("", "_CHOL")
    ).merge(
        fasting,
        on="SEQN",
        how="outer",
        suffixes=("", "_FAST")
    ).merge(
        LBXGH,
        on="SEQN",
        how="outer",
        suffixes=("", "_LBXGH")
    )

# ===================================================================================================
# 👀 Step 3: Validate result
# ===================================================================================================
print("📋 Merge ALL columns:")
print(df.columns.tolist())
print("\n📊 Shape:", df.shape)

# ===================================================================================================
# 💾 Step 14: Save Final Output, Need to change if you are on SCHARE
# ===================================================================================================
OUTPUT_DIR = BASE_DIR / "data" / "processed_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_labs_2003_2004_raw.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Saved to:")
print(OUTPUT_PATH)


# # ===================================================================================================
# # 🧠 Step 7: SEQN Presence Flags states that which SEQN is coming from which file for troubleshooting purpose.
# # ===================================================================================================

# glucose_cpep_insulin["SEQN_in_glucose_cpep_insulin"] = glucose_cpep_insulin["SEQN"].notna()
# chol["SEQN_in_chol"] = chol["SEQN"].notna()
# fasting["SEQN_in_fasting"] = fasting["SEQN"].notna()
# LBXGH["SEQN_in_LBXGH"] = LBXGH["SEQN"].notna()

# df["present_glucose_cpep_insulin"] = df["SEQN_in_glucose_cpep_insulin"].fillna(False)
# df["present_chol"] = df["SEQN_in_chol"].fillna(False)
# df["present_fasting"] = df["SEQN_in_fasting"].fillna(False)
# df["present_LBXGH"] = df["SEQN_in_LBXGH"].fillna(False)
# display(df.head(20))
# print("Unique SEQN count:", df["SEQN"].nunique())
# # ===================================================================================================
# # 🧠 Step 8: Below map will REname the column name for reading purpose. Which may need to recode however, 
# # it has naming convention of "Source_name"_"Actual_Description"
# # ===================================================================================================
# rename_map_glucose_cpep_insulin = {
#     # ID
#     "SEQN": "SEQN",
    
#     # Weights
#     "WTSAF2YR": "WTSAF2YR_fasting_subsample_weight_2yr",
   
#     # Glucose
#     "LBXGLU": "LBXGLU_plasma_glucose_mg_dl",
#     "LBDGLUSI": "LBDGLUSI_plasma_glucose_mmol_l",

#     # C-peptide
#     "LBXCPSI": "LBXCPSI_c_peptide_si_nmol_l",

#     # Insulin
#     "LBXIN": "LBXIN_insulin_uu_ml",
#     "LBDINSI": "LBDINSI_insulin_si_pmol_l"
# }

# # Safe rename (only existing columns)
# df = df.rename(columns={k: v for k, v in rename_map_glucose_cpep_insulin.items() if k in df.columns})
# # =========================
# # 🧬 Cholesterol Rename Map (L13_C)
# # =========================

# rename_map_chol = {
#     "SEQN": "SEQN",

#     # Cholesterol (mg/dL)
#     "LBXTC": "LBXTC_total_cholesterol_mg_dl",
#     "LBXHDD": "LBXHDD_hdl_cholesterol_mg_dl",

#     # Cholesterol (mmol/L)
#     "LBDTCSI": "LBDTCSI_total_cholesterol_mmol_l",
#     "LBDHDDSI": "LBDHDDSI_hdl_cholesterol_mmol_l"
# }

# df = df.rename(columns={k: v for k, v in rename_map_chol.items() if k in df.columns})

# # =========================
# # 🍽️ Fasting Questionnaire Rename Map (PH_C)
# # =========================

# rename_map_phc = {
#     "SEQN": "SEQN",

#     # Coffee / Tea
#     "PHQ020": "PHQ020_coffee_tea_consumption_code",
#     "PHACOFHR": "PHACOFHR_coffee_tea_fast_hours",
#     "PHACOFMN": "PHACOFMN_coffee_tea_fast_minutes",

#     # Alcohol
#     "PHQ030": "PHQ030_alcohol_consumption_code",
#     "PHAALCHR": "PHAALCHR_alcohol_fast_hours",
#     "PHAALCMN": "PHAALCMN_alcohol_fast_minutes",

#     # Gum / mints / cough drops
#     "PHQ040": "PHQ040_gum_mint_cough_consumption_code",
#     "PHAGUMHR": "PHAGUMHR_gum_mint_fast_hours",
#     "PHAGUMMN": "PHAGUMMN_gum_mint_fast_minutes",

#     # Antacids / laxatives
#     "PHQ050": "PHQ050_antacid_laxative_consumption_code",
#     "PHAANTHR": "PHAANTHR_antacid_fast_hours",
#     "PHAANTMN": "PHAANTMN_antacid_fast_minutes",

#     # Dietary supplements
#     "PHQ060": "PHQ060_supplement_consumption_code",
#     "PHASUPHR": "PHASUPHR_supplement_fast_hours",
#     "PHASUPMN": "PHASUPMN_supplement_fast_minutes",

#     # Total fasting duration
#     "PHAFSTHR": "PHAFSTHR_total_fast_hours",
#     "PHAFSTMN": "PHAFSTMN_total_fast_minutes",

#     # Exam session
#     "PHDSESN": "PHDSESN_exam_session"
# }

# df = df.rename(columns={k: v for k, v in rename_map_phc.items() if k in df.columns})


# # ===================================================================================================
# # 👀 Step 3: Validate result
# # ===================================================================================================
# print("📋 Final Columns After Merge + Rename:")
# print(df.columns.tolist())
# print("\n📊 Shape:", df.shape)

# # ===================================================================================================
# # 💾 Step 14: Save Final Output, Need to change if you are on SCHARE
# # ===================================================================================================
# OUTPUT_DIR = BASE_DIR / "data" / "processed_v2"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_labs_2003_2004.csv"

# df.to_csv(OUTPUT_PATH, index=False)

# print("\n✅ Saved to:")
# print(OUTPUT_PATH)


📊 NHANES File Structure Summary:



,file,rows,columns,column_names
0,glucose_cpep_insulin_L10AM_C,3356,7,"[SEQN, WTSAF2YR, LBXGLU, LBDGLUSI, LBXCPSI, LB..."
1,cholesterol_L13_C,8556,5,"[SEQN, LBXTC, LBXHDD, LBDTCSI, LBDHDDSI]"
2,fasting_PH_C,9179,19,"[SEQN, PHQ020, PHACOFHR, PHACOFMN, PHQ030, PHA..."
3,LBXGH_C,6990,2,"[SEQN, LBXGH]"



FILE: glucose_cpep_insulin_L10AM_C
COLUMNS: 7
['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI', 'LBXCPSI', 'LBXIN', 'LBDINSI']

FILE: cholesterol_L13_C
COLUMNS: 5
['SEQN', 'LBXTC', 'LBXHDD', 'LBDTCSI', 'LBDHDDSI']

FILE: fasting_PH_C
COLUMNS: 19
['SEQN', 'PHQ020', 'PHACOFHR', 'PHACOFMN', 'PHQ030', 'PHAALCHR', 'PHAALCMN', 'PHQ040', 'PHAGUMHR', 'PHAGUMMN', 'PHQ050', 'PHAANTHR', 'PHAANTMN', 'PHQ060', 'PHASUPHR', 'PHASUPMN', 'PHAFSTHR', 'PHAFSTMN', 'PHDSESN']

FILE: LBXGH_C
COLUMNS: 2
['SEQN', 'LBXGH']
📋 Merge ALL columns:
['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI', 'LBXCPSI', 'LBXIN', 'LBDINSI', 'LBXTC', 'LBXHDD', 'LBDTCSI', 'LBDHDDSI', 'PHQ020', 'PHACOFHR', 'PHACOFMN', 'PHQ030', 'PHAALCHR', 'PHAALCMN', 'PHQ040', 'PHAGUMHR', 'PHAGUMMN', 'PHQ050', 'PHAANTHR', 'PHAANTMN', 'PHQ060', 'PHASUPHR', 'PHASUPMN', 'PHAFSTHR', 'PHAFSTMN', 'PHDSESN', 'LBXGH']

📊 Shape: (9179, 30)

✅ Saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data